In [1]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import os
from typing import Any, Callable, Dict, Optional, Union, Iterable, Tuple, Sequence
import numpy as np
import pandas as pd


from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))
from stats.masking import mask_feature
from stats.filtering import apply_filters, FilterValue

# k-means 군집 분석 수행
#  - 자동선택 함수(silhouette 기준) 포함
#  - 최적의 k를 찾음, best_k와 k별 결과 요약표 반환
# =========================================================
# 1) k 선택 함수: silhouette 점수 기준으로 최적 k 선택, 
def choose_best_k(
    X_scaled: np.ndarray,
    k_range=range(2, 20),
    random_state=42,
    min_cluster_size=None
) -> tuple[int, pd.DataFrame]:
    rows = []
    best_k = None
    best_score = -1

    for k in k_range:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=30)
        labels = model.fit_predict(X_scaled)

        if min_cluster_size is not None:
            sizes = np.bincount(labels)
            if sizes.min() < min_cluster_size:
                rows.append({
                    "k": k,
                    "silhouette": np.nan,
                    "inertia": float(model.inertia_),
                    "min_cluster_size": int(sizes.min()),
                    "note": "skipped (tiny cluster)"
                })
                continue

        score = silhouette_score(X_scaled, labels)

        rows.append({
            "k": k,
            "silhouette": float(score),
            "inertia": float(model.inertia_),
            "min_cluster_size": int(np.bincount(labels).min()),
            "note": ""
        })

        if score > best_score:
            best_score = score
            best_k = k

    return best_k, pd.DataFrame(rows).sort_values("k").reset_index(drop=True)

# --------------------------------------------------------
# 2) main: 클러스터링 실행 + 요약 만들기
def run_clustering(
    X: pd.DataFrame,
    k: Optional[Union[int, Iterable[int]]] = None,
    random_state=42,
    N_INIT=50,
    min_cluster_size=None,
) -> Tuple[pd.DataFrame, pd.DataFrame, int, Optional[pd.DataFrame], StandardScaler, KMeans]:
    # 1) fit scaler on training data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values)

    # 2) choose k
    k_table = None
    if k is None or (isinstance(k, (list, tuple)) and len(k) > 1):
        best_k, k_table = choose_best_k(
            X_scaled,
            k_range=k,
            random_state=random_state,
            min_cluster_size=min_cluster_size
        )
        k = best_k

    # 3) fit final model
    model = KMeans(n_clusters=k, random_state=random_state, n_init=N_INIT)
    labels = model.fit_predict(X_scaled)

    # 4) output table
    out = X.copy()
    out.insert(0, "cluster", labels)
    out = out.reset_index()

    # 5) cluster summary
    summary = out.groupby("cluster")[X.columns].mean()
    summary.insert(0, "size", out.groupby("cluster").size())

    return out, summary, k, k_table, scaler, model

## 군집 할당 함수
def assign_to_existing_clusters(
    X_new: pd.DataFrame,
    *,
    scaler: StandardScaler,
    model: KMeans,
    train_columns: list[str],
) -> pd.DataFrame:
    """
    Assign new rows to already-fitted clusters.
    Does NOT refit scaler or KMeans.
    """
    X2 = X_new.copy()

    # force same column order as training data
    X2 = X2.reindex(columns=train_columns).fillna(0.0)

    # use training scaler
    X2_scaled = scaler.transform(X2.values)

    # predict existing cluster labels
    labels = model.predict(X2_scaled)

    # distance to each centroid
    distances = model.transform(X2_scaled)
    assigned_distance = [distances[i, lab] for i, lab in enumerate(labels)]

    out = X2.copy()
    out.insert(0, "cluster", labels)
    out.insert(1, "distance_to_assigned", assigned_distance)
    out = out.reset_index()

    return out
# ✅ 옵션1: 파일이 long format인 경우 pivot 스타일로 바꾸기
# 내 파일의 흐름에서는 거의 필수적임.

def build_feature_matrix(
    df: pd.DataFrame,
    *,
    filters: Optional[Dict] = None,
    feature_col: str,
    mask_opts: Optional[Dict] = None,
    outcome_col: str,
    dimension_col: str,
    dimension_set: Optional[Sequence] = None,
    min_nonzero_features: Optional[int] = None,
) -> pd.DataFrame:
    """
    long-format 데이터프레임을 clustering용 feature matrix로 변환한다.

    Parameters
    ----------
    df : pd.DataFrame
        원본 long-format 데이터
    filters : dict | None
        apply_filters에 전달할 필터
    feature_col : str
        사용할 feature 컬럼명 (예: "log_odds")
    mask_opts : dict | None
        mask_feature에 전달할 옵션
    outcome_col : str
        행 index가 될 컬럼명 (예: "V_No")
    dimension_col : str
        열 columns가 될 컬럼명 (예: "outcome_level")
    dimension_set : sequence | None
        열 순서를 고정할 값 목록
    min_nonzero_features : int | None
        0이 아닌 feature 개수가 이 값 이상인 행만 유지
    """
    df2 = df.copy()
    # (1) 필터 적용
    if filters:
        df2 = apply_filters(df2, filters)
    print(f"df에 필터 적용 전: {len(df)}, df에 필터 적용 후: {len(df2)}")
    # (2) 마스킹된 feature 만들기
    if mask_opts:
        df2["_feat"] = mask_feature(df2, feature_col, **mask_opts)
        value_col = "_feat"
    else:
        df2["_feat"] = df2[feature_col]
        value_col = "_feat"

    # (3) 필수 컬럼 확인
    if outcome_col not in df2.columns:
        raise KeyError(f"Missing outcome_col: {outcome_col}")
    if dimension_col not in df2.columns:
        raise KeyError(f"Missing dimension_col: {dimension_col}")
    if value_col not in df2.columns:
        raise KeyError(f"Missing value column: {value_col}")

    # (4) pivot
    X = df2.pivot(index=outcome_col, columns=dimension_col, values=value_col)

    # (5) 열 순서 고정 + 결측 처리
    if dimension_set is not None:
        X = X.reindex(columns=dimension_set)

    X = X.fillna(0.0)

    # (6) 정보 적은 행 제거
    if min_nonzero_features is not None:
        keep = (X != 0).sum(axis=1) >= int(min_nonzero_features)
        X = X.loc[keep]

    return X


# ✅ Option２: 다른 파일에 추가할 데이터가 있는 경우

def build_feature_matrix_from_wide(
    df: pd.DataFrame,
    *,
    filters: Optional[Dict] = None,
    outcome_col: str,
    dimension_set: Optional[Sequence] = None,
    min_nonzero_features: Optional[int] = None,
) -> pd.DataFrame:
    """
    already wide-format 데이터프레임을 clustering용 feature matrix로 정리한다.

    Parameters
    ----------
    df : pd.DataFrame
        원본 wide-format 데이터
    filters : dict | None
        apply_filters에 전달할 필터
    outcome_col : str
        행 index가 될 컬럼명 (예: "V_No")
    dimension_set : sequence | None
        사용할 feature 열 목록 (예: ["책", "허구", "신문", "체험"])
    min_nonzero_features : int | None
        0이 아닌 feature 개수가 이 값 이상인 행만 유지
    """
    df2 = df.copy()

    # (1) 필터 적용
    if filters:
        df2 = apply_filters(df2, filters)

    print(f"df에 필터 적용 전: {len(df)}, df에 필터 적용 후: {len(df2)}")

    # (2) outcome_col 확인
    if outcome_col not in df2.columns:
        raise KeyError(f"Missing outcome_col: {outcome_col}")

    # (3) feature 열 선택
    if dimension_set is None:
        raise ValueError("wide format에서는 dimension_set이 필요함")

    missing_cols = [col for col in dimension_set if col not in df2.columns]
    if missing_cols:
        raise KeyError(f"Missing feature columns: {missing_cols}")

    X = (
        df2[[outcome_col] + list(dimension_set)]
        .copy()
        .set_index(outcome_col)
    )

    # (4) 결측 처리
    X = X.fillna(0.0)

    # (5) 정보 적은 행 제거
    if min_nonzero_features is not None:
        keep = (X != 0).sum(axis=1) >= int(min_nonzero_features)
        X = X.loc[keep]

    return X

# ✅ Option３: 다른 파일에 추가할 데이터가 있는 경우
additional_feature = False
def add_feature_to_matrix(
    X: pd.DataFrame,
    *,
    enabled: bool,
    feature_file: Path,
    outcome_col: str,
    added_outcome_col: str,
    added_feature_col: str,
    added_col_new_name: str,
    added_filters: Optional[Dict] = None,
) -> pd.DataFrame:
    """
    기존 feature matrix X에 외부 feature 1개를 추가한다.

    Parameters
    ----------
    X : pd.DataFrame
        index가 outcome_col(예: V_No)인 feature matrix
    enabled : bool
        추가 feature를 붙일지 여부
    feature_file : Path
        추가 feature가 들어 있는 CSV 파일 경로
    outcome_col : str
        X의 index 이름(예: "V_No")
    added_outcome_col : str
        추가 파일에서 outcome id 컬럼명
    added_feature_col : str
        추가 파일에서 붙일 값 컬럼명
    added_col_new_name : str
        X에 붙인 뒤 새로 사용할 컬럼명
    added_filters : dict | None
        추가 파일에 적용할 필터
    """
    if not enabled:
        return X

    df_added = pd.read_csv(feature_file)

    if added_filters:
        df_added = apply_filters(df_added, added_filters)

    feat = (
        df_added[[added_outcome_col, added_feature_col]]
        .drop_duplicates(subset=[added_outcome_col])
        .set_index(added_outcome_col)
        .rename(columns={added_feature_col: added_col_new_name})
    )

    # X의 index가 이미 outcome_col이라고 가정
    X2 = X.copy()
    X2 = X2.join(feat, how="left").fillna(0.0)

    # index 이름 맞춰주기
    X2.index.name = outcome_col

    return X2

In [2]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

# =========================
# 1) 데이터 경로
# =========================
BASE_DIR = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun")
csv_file = r"csv\cluster_k_means\docu_었결합률.csv"
CSV_PATH = BASE_DIR / Path(csv_file)



# 출력 파일 경로
OUT_DIR = BASE_DIR / "csv" / "cluster_k_means"
os.makedirs(OUT_DIR, exist_ok=True)
   
# 출력 파일 경로들
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
OUT_FEATURE_MATRIX = OUT_DIR / f"feature_matrix_X_{timestamp}.csv"
OUT_K_TABLE_CSV = OUT_DIR / f"k_selection_table_{timestamp}.csv"
OUT_LABEL_WIDE = OUT_DIR / f"cluster_labels_wide_{timestamp}.csv"
OUT_CLUSTERED = OUT_DIR / f"cluster_outcome_level_k{timestamp}.csv"
OUT_SUMMARY = OUT_DIR / f"cluster_summary_k{timestamp}.csv" 

# 파일 읽기
df = pd.read_csv(CSV_PATH, thousands=",")
df.columns = df.columns.str.strip()
print(df.columns)

Index(['docu_id', 'category', 'n_T', 'n_F', 'n', 'T_RATE'], dtype='object')


In [ ]:
# =========================================================
# 2) 설정값들
# =========================================================

# long_format
long_format =False# True   # long_format인 경우 True, 이미 wide_format인 경우 False


# 컬럼명(파일마다 바뀌면 여기만 바꾸면 됨) 
# 데이터 오브젝트 식별자: n개(OUTCOME_COL)의 d-차원(DIMENSION_COL), 각 (n × d) 조합에 피처 값(FEATURE_COL)
OUTCOME_COL = "V_No"    #분석 대상 파일에서 군집화할 대상(행)이 들어가 있는 column명
DIMENSION_COL = "outcome_level" #분석 대상 파일에서각 행을 설명하는 차원(열)이 들어가 있는 column명
FEATURE_COL = "log_odds" #분석 대상 파일에서 데이터값이 들어가 있는 column명

#차원 세트
DIMENSION_SET = ["T비율"]   # 실제 범주 전체

# outcome_level 조건
OUTCOME_MIN = 500
OUTCOME_EXCLUDE = {8999, 3999}

# ✅ 필터는 여기 한 번만
BASE_FILTERS: Dict[str, FilterValue] = {
    #DIMENSION_COL: DIMENSION_SET,
    OUTCOME_COL: lambda s: s >= 1000, #lambda s: (~s.isin(list(OUTCOME_EXCLUDE))),
    #"context_total": lambda s: s >= OUTCOME_MIN,
    # ---- 원하는 필터 있으면 여기 추가 ----
    # "category": ["강의", "낭독"],
    # "outcome_total": lambda s: s >= 500,
}

# 마스킹 옵션(원하면 켜기)
MASK_OPTS = dict(
    p_col="p_value",              # p-value 컬럼명
    p_max=0.05,                    # 예: 0.05, None
    count_col="a_in_context_and_level", # 카운트 컬럼명
    count_min=None,                # 예: 5, 10, None
    abs_min=None,                  # 예: 0.2, 0.5, None
    require_notna=None,            # 예: ["log_odds", "p_value"], None
    fill_value=0.0,                # 마스킹된 값 대체값 (예: 0.0, None  for NaN)
)

# 최소 클러스터 크기: 너무 작은 클러스터는 건너뜀 (선택)
MIN_CLUSTER_SIZE: Optional[int] = None   # 예: 5, 10, None

# outcome_level(행) 단위 추가 필터: 0 아닌 feature가 너무 적으면 제거
MIN_NONZERO_FEATURES: Optional[int] = 2    # 예: 3, 4, None

# k 여러 개를 한 번에 + k마다 여러 번 반복
K_VALUES = list(range(2, 9))   # 2~20
N_REPEATS_PER_K = 30
BASE_RANDOM_STATE = 42


In [ ]:
# =========================================================
#  main: 두 단계 실행 A 높은 빈도 대상으로 군집화 기준 만들기. B 전체 적용
# =========================================================

# ----------------------------------------------
# Step 1. 고빈도만으로 모델 학습
# ----------------------------------------------

# high-frequency training set
TRAIN_FILTERS = {
    #OUTCOME_COL: lambda s: ~s.isin([8999, 3999]),
    "total": lambda s: s >= 500,
}

if long_format:
    X_train = build_feature_matrix(
        df,
        filters=(BASE_FILTERS | TRAIN_FILTERS),
        feature_col=FEATURE_COL,
        mask_opts=MASK_OPTS,
        outcome_col=OUTCOME_COL,
        dimension_col=DIMENSION_COL,
        dimension_set=DIMENSION_SET,
        min_nonzero_features=MIN_NONZERO_FEATURES,
    )
else:
    X_train = build_feature_matrix_from_wide(
        df,
        filters=(BASE_FILTERS | TRAIN_FILTERS),
        outcome_col=OUTCOME_COL,
        dimension_set=DIMENSION_SET,
        min_nonzero_features=MIN_NONZERO_FEATURES,
    )
        

clustered_train, summary_train, chosen_k, k_table, scaler, model = run_clustering(
    X_train,
    k=K_VALUES,
    random_state=BASE_RANDOM_STATE,
    N_INIT=50,
    min_cluster_size=MIN_CLUSTER_SIZE,
)
# 4) 결과 저장
if k_table is not None:
    print("\n[TRAIN 클러스터링 k별 요약]")
    print(k_table)
    print(f"\n[TRAIN 선택된 최적 k: {chosen_k}]")
    k_table.to_csv(OUT_K_TABLE_CSV, index=False, encoding="utf-8-sig")
    print(f"[TRAIN k 선택표 저장 완료: {OUT_K_TABLE_CSV}]")

print(f"\n[TRAIN 최종 선택된 k: {chosen_k}]")
print(f"[TRAIN 클러스터링 결과 요약]")
print(summary_train)


df에 필터 적용 전: 3265, df에 필터 적용 후: 126


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.

In [89]:

# ----------------------------------------------
# Step B. 전체 동사 feature matrix 만들기
# ----------------------------------------------


if long_format:
    X_all = build_feature_matrix(
        df,
        filters=BASE_FILTERS,
        feature_col=FEATURE_COL,
        mask_opts=MASK_OPTS,
        outcome_col=OUTCOME_COL,
        dimension_col=DIMENSION_COL,
        dimension_set=DIMENSION_SET,
        min_nonzero_features=None,
    )
else:
    X_all = build_feature_matrix_from_wide(
        df,
        filters=(BASE_FILTERS),
        outcome_col=OUTCOME_COL,
        dimension_set=DIMENSION_SET,
        min_nonzero_features=MIN_NONZERO_FEATURES,
    )
        
# ----------------------------------------------
# Step 3. 전체 assign
# ----------------------------------------------

assigned_all = assign_to_existing_clusters(
    X_all,
    scaler=scaler,
    model=model,
    train_columns=X_train.columns.tolist(),
)
train_ids = set(X_train.index)

assigned_all["source"] = assigned_all[OUTCOME_COL].apply(
    lambda x: "train" if x in train_ids else "assigned"
)

summary_all = assigned_all.groupby("cluster")[X_train.columns].mean()
summary_all["size"] = assigned_all.groupby("cluster").size()
print(f"\n[전체 클러스터링 결과 요약]")
print(summary_all)

df에 필터 적용 전: 3265, df에 필터 적용 후: 3237

[전체 클러스터링 결과 요약]
              T비율      T->F      F->T  size
cluster                                    
0        0.297965  0.403053  0.144997  1139
1        0.645616  0.117226  0.397795  1845


In [90]:
assigned_all

,V_No,cluster,distance_to_assigned,T비율,T->F,F->T,source
0,1001,0,0.332447,0.331072,0.322865,0.128448,train
1,1101,0,0.799064,0.340909,0.435294,0.200000,assigned
2,1102,0,0.208165,0.307419,0.381791,0.142968,train
3,1105,0,1.420228,0.350000,0.500000,0.250000,assigned
4,1201,1,4.768145,0.750000,0.333333,1.000000,assigned
...,...,...,...,...,...,...,...
2979,7644,1,0.499942,0.700000,0.142857,0.333333,assigned
2980,7645,0,2.576242,0.181818,0.666667,0.125000,assigned
2981,7646,1,5.284387,0.600000,0.500000,1.000000,assigned
2982,8110,0,1.065899,0.428571,0.333333,0.250000,assigned


In [91]:
OUT_CLUSTERED_TRAIN = OUT_DIR / f"{OUTCOME_COL}_cluster_train_{timestamp}.csv"
OUT_SUMMARY_TRAIN   = OUT_DIR / f"{OUTCOME_COL}_cluster_summary_train_{timestamp}.csv"
OUT_ASSIGNED_ALL    = OUT_DIR / f"{OUTCOME_COL}_cluster_assigned_all_{timestamp}.csv"
OUT_LABELS_ALL      = OUT_DIR / f"{OUTCOME_COL}_cluster_labels_all_{timestamp}.csv"

# 1. training verbs only
clustered_train.to_csv(OUT_CLUSTERED_TRAIN, index=False, encoding="utf-8-sig")
summary_train.to_csv(OUT_SUMMARY_TRAIN, encoding="utf-8-sig")

# 2. all verbs assigned to existing clusters
assigned_all.to_csv(OUT_ASSIGNED_ALL, index=False, encoding="utf-8-sig")

# 3. compact label file
labels_all = assigned_all[[OUTCOME_COL, "cluster", "distance_to_assigned", "source"]].copy()
labels_all.to_csv(OUT_LABELS_ALL, index=False, encoding="utf-8-sig")

print(
    f"\n[저장 완료]\n"
    f"{OUT_CLUSTERED_TRAIN}\n"
    f"{OUT_SUMMARY_TRAIN}\n"
    f"{OUT_ASSIGNED_ALL}\n"
    f"{OUT_LABELS_ALL}"
)


[저장 완료]
C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\V_No_cluster_train_2026-03-30_23-02.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\V_No_cluster_summary_train_2026-03-30_23-02.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\V_No_cluster_assigned_all_2026-03-30_23-02.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\V_No_cluster_labels_all_2026-03-30_23-02.csv


In [ ]:
# ✅ Option: 추가할 데이터가 있는 경우
additional_feature = False  # 예: True/False 토글
ADDED_FEATURE_FILE = BASE_DIR / "csv" / "통계용" / "2025-12-27_11-49_logodds_EP_T_by_V_No_ID.csv"
ADDED_OUTCOME_COL = "V_No"
ADDED_FEATURE_COL = "log_odds"
ADDED_FILTERS = {"outcome_level": True}
ADDED_COL_NEW_NAME = "EP_T"  # 추가할 피처명

In [38]:
# =========================================================
#  main: 한 번만 적용
# =========================================================

# ----------------------------------------------------

print(f"[클러스터링 시작]: {CSV_PATH}, {timestamp} \n")


# 2) feature matrix 만들기, 필터/마스킹 포함
X = build_feature_matrix(
    df,
    filters=BASE_FILTERS,
    feature_col=FEATURE_COL,
    mask_opts=MASK_OPTS,
    outcome_col=OUTCOME_COL,
    dimension_col=DIMENSION_COL,
    dimension_set=DIMENSION_SET,
    min_nonzero_features=MIN_NONZERO_FEATURES,
)
if X.shape[0] < 2:
    raise ValueError(f"필터/전처리 후 {OUTCOME_COL}이 너무 적음. (최소 2개 필요)")
X.to_csv(OUT_FEATURE_MATRIX, encoding="utf-8-sig")     # 참고용: feature matrix 저장
print(f"[feature matrix 생성 완료]: {X.shape}, {OUT_FEATURE_MATRIX}")

# 2-1) 추가 피처 합치기
if additional_feature == True:
    X = add_feature_to_matrix(
        X,
        enabled=additional_feature,
        feature_file=ADDED_FEATURE_FILE,
        outcome_col=OUTCOME_COL,
        added_outcome_col=ADDED_OUTCOME_COL,
        added_feature_col=ADDED_FEATURE_COL,
        added_col_new_name=ADDED_COL_NEW_NAME,
        added_filters=ADDED_FILTERS,
    )

# 3) clustering

clustered, summary, chosen_k, k_table, scaler, model = run_clustering(
    X,
    k=K_VALUES,
    random_state=BASE_RANDOM_STATE,
    N_INIT=50,
    min_cluster_size=MIN_CLUSTER_SIZE,
)
# 4) 결과 저장
if k_table is not None:
    print("\n[클러스터링 k별 요약]")
    print(k_table)
    print(f"\n[선택된 최적 k: {chosen_k}]")
    k_table.to_csv(OUT_K_TABLE_CSV, index=False, encoding="utf-8-sig")
    print(f"[k 선택표 저장 완료: {OUT_K_TABLE_CSV}]")

print("\n[Done]")
print(f"[최종 선택된 k: {chosen_k}]")
print(f"[클러스터링 결과 요약]")
print(summary)


[클러스터링 시작]: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\결과값\tense\A_oz_문서범주_by_V_No_2026-03-28_06-52.csv, 2026-03-28_11-19 

df에 필터 적용 전: 13080, df에 필터 적용 후: 13072
[feature matrix 생성 완료]: (1679, 4), C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\feature_matrix_X_2026-03-28_11-19.csv

[클러스터링 k별 요약]
   k  silhouette      inertia  min_cluster_size note
0  2    0.388122  4047.656011               742     
1  3    0.299163  3388.414854               318     
2  4    0.325814  2803.419677               108     
3  5    0.312554  2360.822952               107     
4  6    0.321634  2053.572158                55     
5  7    0.349319  1747.101017                49     
6  8    0.329627  1614.009186                48     

[선택된 최적 k: 2]
[k 선택표 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\k_selection_table_2026-03-28_11-19.csv]

[Done]
[최종 선택된 k: 2]
[클러스터링 결과 요약]
outcome_level  size         책        허구        신문     

In [11]:
# 파일로 결과 저장

clustered.to_csv(OUT_CLUSTERED, index=False, encoding="utf-8-sig")
summary.to_csv(OUT_SUMMARY, encoding="utf-8-sig")
print(f"\n[클러스터링 결과 저장 완료: \n{OUT_CLUSTERED}, \n{OUT_SUMMARY}]")
# wide 형식으로 라벨 저장
clustered_wide = clustered.reset_index().pivot(
    index=OUTCOME_COL,
    columns="cluster",
    values="cluster"
)
clustered_wide.columns = [f"cluster_{col}" for col in clustered_wide.columns]
clustered_wide.reset_index(inplace=True)
clustered_wide.to_csv(OUT_LABEL_WIDE, index=False, encoding="utf-8-sig")
print(f"[클러스터 라벨(wide) 저장 완료: {OUT_LABEL_WIDE}]")




[클러스터링 결과 저장 완료: 
C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\cluster_outcome_level_k2026-03-28_10-43.csv, 
C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\cluster_summary_k2026-03-28_10-43.csv]
[클러스터 라벨(wide) 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\cluster_labels_wide_2026-03-28_10-43.csv]


In [ ]:
# =========================================================
# 6) 추가 분석: 특정 군집만 다시 군집화
# =========================================================

# 1차 결과 (clustered는 outcome_level + cluster + 12피처)
sub = clustered[clustered["cluster"] == 0].copy()   # 예: 1번 군집만 다시 보기

# 2차 군집 입력 X2: ID를 index로 둠
# 추가한 열이 있는 경우
if additional_feature == True:  
    X2 = sub.set_index(OUTCOME_COL)[DIMENSION_SET+[ADDED_COL_NEW_NAME]]
    
# 추가한 열이 없는 경우
else:   
    X2 = sub.set_index(OUTCOME_COL)[DIMENSION_SET]

# 2차 군집
clustered2, summary2, chosen_k2, k_table2 = run_clustering(
    X2, k=9, random_state=42, N_INIT=100
)

print("\n[2차 군집화 완료]")
if k_table2 is not None:
    print("\n[2차 클러스터링 k별 요약]")
    print(k_table2)
    print(f"\n[2차 선택된 최적 k: {chosen_k2}]") 



[2차 군집화 완료]


In [ ]:
OUT_CLUSTERED = OUT_DIR / f"cluster_outcome_level2_{timestamp}.csv"
OUT_SUMMARY = OUT_DIR / f"cluster_summary2_{timestamp}.csv" 

clustered2.to_csv(OUT_CLUSTERED, index=False, encoding="utf-8-sig")
summary2.to_csv(OUT_SUMMARY, encoding="utf-8-sig")
print(f"\n[클러스터링 결과 저장 완료: \n{OUT_CLUSTERED}, \n{OUT_SUMMARY}]")
print(summary2)


[클러스터링 결과 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\cluster_outcome_level2_2025-12-27_12-25.csv, C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\cluster_k_means\cluster_summary2_2025-12-27_12-25.csv]
         size       101       102       111       112       123       122  \
cluster                                                                     
0          45  0.638764 -3.317157  1.252431  1.628615  0.069276 -0.450177   
1          24  0.152193 -2.990369  0.438010 -0.007504  0.174501  0.209818   
2          32 -0.304998  1.743092  0.585265  0.225417 -1.345900  0.049787   
3          59  0.988199 -3.927155  0.762510 -0.825700 -1.327545 -0.648122   
4          22  0.356544 -1.908404 -2.693098 -2.557221 -2.769581 -0.231157   
5          17  0.543876 -2.638357 -0.191047  0.433904  0.120431  0.818207   
6          38  0.168032 -3.090605  0.536664  0.318075  2.215079 -0.057742   
7          32 -0.454251 -0.775056 -0.678813  0.678754 -